In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPVisionModel

from fake_image_detection.config import Config
from fake_image_detection.dataset import load_dataloaders

from torch import nn

In [ ]:
class FakeImageDetector(nn.Module):

    def __init__(self, config: Config):
        super().__init__()
        self.model = CLIPVisionModel.from_pretrained(config.pretrained_model_name)
        self.head = nn.Linear(config.embedding_size, config.num_classes)
        # only train the head
        for param in self.model.parameters():
            param.requires_grad = False

    def __call__(self, image: torch.Tensor) -> torch.Tensor:
        output = self.model.vision_model(image)
        return self.head(output.pooler_output)


model = FakeImageDetector(Config())

In [ ]:
train_dataloader, val_dataloader, test_dataloader = load_dataloaders(Config)

In [ ]:
optimiser = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_function = nn.CrossEntropyLoss()

for image, label in train_dataloader:
    pred = model(image)
    softmax_pred = nn.functional.softmax(pred, dim=1)

    # loss for batch
    loss = loss_function(pred, label)
    print(loss)

    loss.backward()
    optimiser.step()
    optimiser.zero_grad()
